# DHIS2 Climate Tools – Malaria Gridding & Population Weighting

This notebook demonstrates how DHIS2 Climate Tools can be used to
transform **facility-based malaria data** into **population-weighted
gridded risk surfaces** for Malawi.

The workflow reflects the use case where malaria cases are recorded at
health facilities, while transmission occurs within facility catchment
areas.

------------------------------------------------------------------------

## 1. Objective

-   Pull malaria case data directly from **DHIS2** using Climate Tools
-   Interpolate facility-level cases into a **continuous spatial grid**
-   Mask non-inhabited areas (e.g. **water bodies**)
-   Overlay **population distribution** to generate population-weighted
    malaria surfaces
-   Produce outputs suitable for climate–health analysis and EWARS

------------------------------------------------------------------------

## 2. Import Required Libraries


In [ ]:
from io import StringIO
import pandas as pd
import xarray as xr
import geopandas as gpd

from preparedata import prepare_data
from gridding import linear_grid
from masking import mask
from plot import plotData
from dhis2eo.data.worldpop import pop_total

## 3. Pull Malaria Case Data from DHIS2

Facility-level malaria cases are retrieved directly from DHIS2 by
specifying the instance URL, reporting period, data dimension, and
organisation unit level.

In [ ]:
data_str = prepare_data(
    base_url="some url",
    username="username",
    password="password",
    dx='jPEcKbn7jmh',
    pe="202501",
    ou_level="4"
)

dataValues = pd.read_csv(StringIO(data_str))

Convert the returned CSV string into a pandas DataFrame:

In [ ]:
dataValues = pd.read_csv(StringIO(data_str))

## 2. Linear Gridding

Facility point data are interpolated into a continuous spatial surface
using **linear interpolation**.

In [ ]:
lin = linear_grid(dataValues)

## 3. Population Data and Weights

Population data is being downloaded from the world population dataset by specifying the year and country code

In [ ]:
country_code = 'MWI'
pop_ds = pop_total.get("2025", country_code)

pop_da = pop_ds['total_pop'].rename({'x': 'lon', 'y': 'lat'})
rst = pop_da.reindex_like(lin, method="nearest")

total_pop_sum = rst.sum(dim=("lat", "lon"))
weights = rst / total_pop_sum

## 4. Population-Weighted Redistribution

In [ ]:
total_cases_val = lin['cases'].isel(time=0).sum(dim=("lat", "lon"))

redistributed_da = weights * total_cases_val
cases_ds = redistributed_da.to_dataset(name="cases")
cases_ds

## 5. Masking and Visualization

In [ ]:
districts_path = r"C:\\Users\\ShnkMn\\Documents\\CMS\\climate-tools\\docs\\data\\Districts.shp"
overlay = gpd.read_file(districts_path).to_crs(epsg=4326)

grd_masked = mask(lin, districts_path)
msk_redistributed = mask(cases_ds, districts_path)

print("Masked Redistributed Data:")
print(msk_redistributed)

plotData(grd_masked, overlay)
plotData(msk_redistributed.squeeze(), overlay)